# Lakehouse — Başlangıç Notebook'u

Bu pod, lakehouse bağlantı bilgileriniz önceden enjekte edilmiş şekilde açıldı — ek
konfigürasyon gerekmez. Üç sorgu motoru da hazır, hepsi `lakehouse_nb` yardımcı modülü
üzerinden (`import lakehouse_nb as lh`):

- **PyIceberg** (`lh.iceberg()`) — REST katalog üzerinden doğrudan Python/pandas okuma.
- **Spark** (`lh.spark()`) — `lakehouse` (prod Silver) ve `rawlake` katalogları salt-okuma;
  kendi `sandbox` kataloğunuzun kişisel şemasına (`lh.sandbox_schema()`, örn.
  "sandbox.<kullanıcı-adınız>") okuma/yazma — `lh.spark()` bunu zaten varsayılan
  şema/katalog olarak ayarlar.
- **Trino** (`lh.trino().cursor()`) — hızlı SQL, salt-okuma (`lakehouse` kataloğu); sandbox
  CTAS için `lh.sandbox_schema()`'yı fully-qualified tablo adında kullanın.

Aşağıdaki tablo adları **placeholder**'dır (`<ns>.<tablo>`) — bu notebook ürün-generic'tir,
belirli bir demo şemasına bağlı değildir. Gerçek şema/tablo adlarınızı görmek için en alttaki
"kendi şema ve tablolarını listele" hücresini çalıştırın.

In [ ]:
import lakehouse_nb as lh

print(lh.sandbox_schema())  # kişisel sandbox şemanız, örn. "sandbox.<kullanıcı-adınız>"

## 1) PyIceberg — REST katalogdan doğrudan oku

`<ns>.<tablo>` yerine kendi namespace/tablo adınızı yazın (aşağıdaki listeleme hücresiyle
bulabilirsiniz).

In [ ]:
cat = lh.iceberg()
cat.list_namespaces()

In [ ]:
# Placeholder: <ns>.<tablo> -> kendi namespace/tablo adınızla değiştirin
cat.load_table("<ns>.<tablo>").scan().to_pandas().head()

## 2) Spark — prod Silver'ı oku, kendi sandbox şemana yaz

`lakehouse` ve `rawlake` katalogları salt-okuma prod verisidir. Kendi denemeleriniz için
kişisel sandbox şemanıza (`lh.sandbox_schema()`, aşağıdaki yorumlu örnek) yazabilirsiniz —
`lh.spark()` bu şemayı önceden `CREATE SCHEMA IF NOT EXISTS` ile açar ve oturumun varsayılan
katalog/şeması olarak ayarlar.

In [ ]:
s = lh.spark()
s.sql("SHOW SCHEMAS IN lakehouse").show()

In [ ]:
# Placeholder: <ns>.<tablo> -> kendi namespace/tablo adınızla değiştirin.
# Yazım örneği (varsayılan olarak yorum satırı -- lh.spark() sizi zaten kendi sandbox
# şemanıza (lh.sandbox_schema()) USE'lamış durumda bırakır, o yüzden aşağıdaki bare
# CREATE TABLE/saveAsTable oraya yazar; fully-qualified "lakehouse.<ns>.<tablo>" gibi
# referanslar bundan etkilenmez):
#
# df = s.table("lakehouse.<ns>.<tablo>")
# df.writeTo("<tablo>").createOrReplace()  # -> kişisel sandbox şemanıza yazar

## 3) Trino — hızlı SQL (salt-okuma)

In [ ]:
# Placeholder: <ns>.<tablo> -> kendi namespace/tablo adınızla değiştirin
c = lh.trino().cursor()
c.execute("SELECT count(*) FROM lakehouse.<ns>.<tablo>")
c.fetchone()

## 4) Kendi şema/tablolarınızı keşfedin

Yukarıdaki hücrelerdeki `<ns>.<tablo>` placeholder'larını gerçek adlarla değiştirmeden önce,
hangi şema ve tabloların var olduğunu görmek için bu hücreyi çalıştırın (Spark veya Trino ile).

In [ ]:
# Spark ile:
s.sql("SHOW SCHEMAS IN lakehouse").show()
s.sql("SHOW TABLES IN lakehouse.<ns>").show()

# Trino ile (eşdeğer):
# c.execute("SHOW SCHEMAS FROM lakehouse"); c.fetchall()
# c.execute("SHOW TABLES FROM lakehouse.<ns>"); c.fetchall()